# 3 — Activities

**Theme:** labelling periods of a record so you can analyse them separately.

An *activity* is a named set of time periods — a task, a process step, a
background period. Once marked, activities drive the statistics in
[4 — Statistics and exposure](04-statistics-and-exposure.ipynb) and the shading
in [5 — Plotting](05-plotting.ipynb).

Activities are stored in absolute time, so the same definitions can be applied
to every instrument in a campaign.

In [ ]:
import aerosoltools as at

smps = at.load_smps_file("../../tests/data/Sample_SMPS.txt")
smps.activities

Every dataset starts with a single built-in activity, `"All data"`, covering the
whole record.

## Marking known periods

The usual case: you know when each task ran. Pass a dictionary of activity name
to a list of `(start, end)` pairs — an activity can have several occurrences.

In [ ]:
activity_periods = {
    "Emission": [
        ("2018-02-27 10:18:00", "2018-02-27 10:31:00"),
        ("2018-02-27 10:35:00", "2018-02-27 10:48:00"),
        ("2018-02-27 10:52:00", "2018-02-27 11:30:00"),
        ("2018-02-27 12:39:00", "2018-02-27 12:48:00"),
    ],
    "Constant phase 1": [("2018-02-27 11:43:00", "2018-02-27 12:35:00")],
    "Constant phase 2": [("2018-02-27 12:55:00", "2018-02-27 13:45:00")],
    "Background": [("2018-02-27 13:48:00", "2018-02-27 14:39:00")],
}

smps.mark_activities(activity_periods)
smps.activities

`.activity_periods` gives the periods back, as absolute timestamps.

In [ ]:
smps.activity_periods["Background"]

## Marking by threshold

When you do not have a task log, `mark_threshold` labels every sample on one
side of a value.

In [ ]:
cpc = at.load_cpc_file("../../tests/data/Sample_CPC_AIM.txt")
cpc.mark_threshold("Elevated", threshold=20000, threshold_direction="above")
cpc.activities

## Finding peaks

`peak_finder` marks samples that rise above a rolling baseline — useful for
picking out short emission events without knowing when they happened. It flags
a sample when the signal exceeds `baseline + ratio * rolling_std`, where the
baseline is a rolling median over `window` samples.

In [ ]:
cpc.peak_finder(window=15, ratio=2.5, method="median")
cpc.activities

## Renaming

`rename_activity` keeps the periods and the mask, changing only the label.

In [ ]:
cpc.rename_activity("Elevated", "Above 20k")
cpc.activities

## Getting the data back out

`get_activity_data` returns only the rows inside an activity's periods — across
all of its occurrences.

In [ ]:
emission = smps.get_activity_data("Emission")
background = smps.get_activity_data("Background")

print(f"Emission   : {len(emission)} samples")
print(f"Background : {len(background)} samples")
print(f"All data   : {len(smps.get_activity_data('All data'))} samples")

`get_activity_extra_data` does the same for the auxiliary channels in
`.extra_data`.

In [ ]:
cpc.get_activity_extra_data("Above 20k").head()

Because the result is a plain DataFrame, ordinary pandas works from here.

In [ ]:
emission["Total_conc"].describe()

That is the manual route. The next notebook does this across every activity at
once, with the statistics that matter for exposure assessment.

---

**Next:** [4 — Statistics and exposure](04-statistics-and-exposure.ipynb).